# Benchmark Comparison

Structured review of benchmark results.

Fair automatic-evaluation source:

- Prefer `metrics_fair.csv` when present. It excludes infrastructure rows and applies local post-hoc rescore results from saved transcripts.
- Raw `metrics_all.csv` remains the immutable merged-run artifact, but it should not drive model-quality summaries after audit/rescore.

Comparability rules:

- Strict comparisons only happen within the same `capability_mode`, `telemetry_trust`, and `provider_backend` cohort.
- `raw_api` is a `single_shot` baseline. Agent harnesses are `agent_iterated` runs.
- The all-harness chart is descriptive only; do not use it as a strict ranking.
- `model` is the canonical comparison label. `adapter_model` / `provider_backend` / `api_backend` / `pricing_model` explain what actually served and priced the run.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RESULTS = Path('results/full_combined_v3')
BASELINE_RESULTS = None  # Set to Path('...') for an explicit baseline comparison.
PREFERRED_METRICS = 'metrics_fair.csv'
RAW_METRICS = 'metrics_all.csv'
CSVS = ['metrics_springboot.csv', 'metrics_angular.csv', 'metrics_react.csv', 'metrics_data.csv', RAW_METRICS]
OPENCODE_GO_IDS = {
    'glm-5.2', 'glm-5.1', 'kimi-k2.7-code', 'kimi-k2.6', 'deepseek-v4-pro',
    'deepseek-v4-flash', 'mimo-v2.5', 'mimo-v2.5-pro', 'minimax-m3',
    'minimax-m2.7', 'qwen3.7-max', 'qwen3.7-plus', 'qwen3.6-plus',
}
REQUIRED_COLUMNS = [
    'harness', 'task', 'model', 'adapter_model', 'provider_backend', 'api_backend',
    'pricing_model', 'run', 'capability_mode', 'telemetry_trust', 'tool_set',
    'build_ok', 'test_ok', 'in_tok', 'out_tok', 'cost_usd', 'model_calls',
    'telemetry_note', 'latency_s', 'workdir', 'transcript_path', 'error',
    'fair_build_ok', 'fair_test_ok', 'fair_included', 'fair_status', 'fair_notes',
]

def opencode_go_selector(model):
    model = '' if pd.isna(model) else str(model)
    if model.startswith('opencode-go/'):
        return model
    suffix = model.split('/')[-1]
    return f'opencode-go/{suffix}' if suffix in OPENCODE_GO_IDS else ''

def truthy(series):
    return series.fillna('').astype(str).str.strip().str.lower().isin({'true', '1'})

def ensure_schema(frame):
    frame = frame.copy()
    for column in REQUIRED_COLUMNS:
        if column not in frame.columns:
            frame[column] = ''
    frame['harness'] = frame['harness'].fillna('').replace('', 'raw_api')
    frame['telemetry_note'] = frame['telemetry_note'].fillna('')
    missing_adapter = frame['adapter_model'].fillna('').eq('')
    agent_or_go = frame['harness'].ne('raw_api') | frame['telemetry_note'].str.contains('opencode_go', na=False)
    frame.loc[missing_adapter & agent_or_go, 'adapter_model'] = frame.loc[missing_adapter & agent_or_go, 'model'].map(opencode_go_selector)
    still_missing = frame['adapter_model'].fillna('').eq('')
    frame.loc[still_missing, 'adapter_model'] = frame.loc[still_missing, 'model']

    missing_provider = frame['provider_backend'].fillna('').eq('')
    is_go = frame['adapter_model'].astype(str).str.startswith('opencode-go/') | frame['telemetry_note'].str.contains('opencode_go', na=False)
    frame.loc[missing_provider & is_go, 'provider_backend'] = 'opencode-go'
    missing_provider = frame['provider_backend'].fillna('').eq('')
    frame.loc[missing_provider, 'provider_backend'] = frame.loc[missing_provider, 'adapter_model'].astype(str).str.split('/').str[0]

    missing_api = frame['api_backend'].fillna('').eq('')
    frame.loc[missing_api & frame['telemetry_note'].str.contains('opencode_go_chat', na=False), 'api_backend'] = 'opencode_go_chat'
    missing_api = frame['api_backend'].fillna('').eq('')
    frame.loc[missing_api & frame['telemetry_note'].str.contains('openrouter_usage', na=False), 'api_backend'] = 'openrouter_chat'
    missing_api = frame['api_backend'].fillna('').eq('')
    frame.loc[missing_api & frame['harness'].eq('raw_api') & frame['provider_backend'].eq('opencode-go'), 'api_backend'] = 'opencode_go_chat'
    missing_api = frame['api_backend'].fillna('').eq('')
    frame.loc[missing_api & frame['harness'].eq('raw_api'), 'api_backend'] = 'openrouter_chat'
    missing_api = frame['api_backend'].fillna('').eq('')
    frame.loc[missing_api & frame['harness'].isin(['omp', 'opencode', 'hermes']), 'api_backend'] = frame.loc[missing_api & frame['harness'].isin(['omp', 'opencode', 'hermes']), 'harness'] + '_cli'

    missing_price = frame['pricing_model'].fillna('').eq('')
    frame.loc[missing_price & frame['provider_backend'].eq('opencode-go'), 'pricing_model'] = frame.loc[missing_price & frame['provider_backend'].eq('opencode-go'), 'adapter_model'].map(opencode_go_selector)
    still_missing_price = frame['pricing_model'].fillna('').eq('')
    frame.loc[still_missing_price, 'pricing_model'] = frame.loc[still_missing_price, 'adapter_model']

    frame['capability_mode'] = frame['capability_mode'].fillna('')
    frame.loc[frame['capability_mode'].eq('') & frame['harness'].eq('raw_api'), 'capability_mode'] = 'single_shot'
    frame.loc[frame['capability_mode'].eq(''), 'capability_mode'] = 'agent_iterated'
    frame['telemetry_trust'] = frame['telemetry_trust'].fillna('')
    frame.loc[frame['telemetry_trust'].eq('') & frame['harness'].eq('raw_api'), 'telemetry_trust'] = 'exact'
    frame.loc[frame['telemetry_trust'].eq('') & frame['harness'].isin(['omp', 'opencode']), 'telemetry_trust'] = 'parsed'
    frame.loc[frame['telemetry_trust'].eq('') & frame['harness'].eq('hermes'), 'telemetry_trust'] = 'blank'
    return frame

def selected_csv_names(path):
    # Prefer the audited/rescored fair table for model-quality summaries.
    if (path / PREFERRED_METRICS).exists():
        return [PREFERRED_METRICS]
    if (path / RAW_METRICS).exists():
        return [RAW_METRICS]
    return [name for name in CSVS if name != RAW_METRICS]

def load_result_dir(path, label):
    # Prefer a single consolidated CSV. Reading per-stack CSVs as well would double-count merged runs.
    frames = []
    for csv_name in selected_csv_names(path):
        csv_path = path / csv_name
        if csv_path.exists():
            part = pd.read_csv(csv_path)
            part['source_csv'] = csv_name
            frames.append(part)
    if not frames:
        raise FileNotFoundError(f'No metrics CSVs found under {path}')
    data = ensure_schema(pd.concat(frames, ignore_index=True))
    data['result_set'] = label
    return data

def finalize(frame):
    frame = frame.copy()
    for column in ['build_ok', 'test_ok', 'fair_build_ok', 'fair_test_ok', 'fair_included']:
        frame[column] = frame[column].fillna('').astype(str).str.strip()
    raw_without_fair = frame['fair_included'].eq('')
    frame.loc[raw_without_fair, 'fair_included'] = 'True'
    frame.loc[frame['fair_build_ok'].eq(''), 'fair_build_ok'] = frame.loc[frame['fair_build_ok'].eq(''), 'build_ok']
    frame.loc[frame['fair_test_ok'].eq(''), 'fair_test_ok'] = frame.loc[frame['fair_test_ok'].eq(''), 'test_ok']
    frame.loc[frame['fair_status'].fillna('').astype(str).str.strip().eq(''), 'fair_status'] = 'raw_metrics_all'
    for column in ['in_tok', 'out_tok', 'cost_usd', 'model_calls', 'latency_s', 'run']:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    frame['included_for_quality'] = truthy(frame['fair_included'])
    frame['passed'] = frame['included_for_quality'] & frame['fair_build_ok'].eq('True') & frame['fair_test_ok'].eq('True')
    frame['failed'] = frame['included_for_quality'] & (frame['fair_build_ok'].eq('False') | frame['fair_test_ok'].eq('False'))
    frame['errored'] = frame['included_for_quality'] & (frame['fair_build_ok'].eq('ERROR') | frame['fair_test_ok'].eq('ERROR') | frame['error'].fillna('').astype(str).str.strip().ne(''))
    frame['excluded'] = ~frame['included_for_quality']
    frame['model_short'] = frame['model'].astype(str).str.split('/').str[-1]
    frame['adapter_model_short'] = frame['adapter_model'].astype(str).str.split('/').str[-1]
    frame['cohort'] = frame['capability_mode'] + ' / ' + frame['telemetry_trust'] + ' / ' + frame['provider_backend']
    return frame

all_df = finalize(load_result_dir(RESULTS, 'current'))
df = all_df[all_df['included_for_quality']].copy()
excluded_df = all_df[all_df['excluded']].copy()
if BASELINE_RESULTS:
    baseline_all_df = finalize(load_result_dir(BASELINE_RESULTS, 'baseline'))
    baseline_df = baseline_all_df[baseline_all_df['included_for_quality']].copy()
else:
    baseline_all_df = pd.DataFrame(columns=all_df.columns)
    baseline_df = pd.DataFrame(columns=df.columns)

print(f'{len(all_df)} current rows loaded from {RESULTS} / {sorted(all_df["source_csv"].dropna().unique())}')
print(f'{len(df)} scored rows; {len(excluded_df)} excluded infrastructure/unscored rows')
print('Harnesses:', sorted(df['harness'].dropna().unique()))
print('Models:', sorted(df['model'].dropna().unique()))
print('Adapter models:', sorted(df['adapter_model'].dropna().unique()))
print('Provider backends:', sorted(df['provider_backend'].dropna().unique()))
print('API backends:', sorted(df['api_backend'].dropna().unique()))
print('Cohorts:', sorted(df['cohort'].dropna().unique()))


396 current rows loaded from results/full_combined_v3 / ['metrics_fair.csv']
275 scored rows; 121 excluded infrastructure/unscored rows
Harnesses: ['hermes', 'omp', 'opencode', 'raw_api']
Models: ['deepseek/deepseek-v4-flash', 'minimax/minimax-m3', 'z-ai/glm-5.2']
Adapter models: ['opencode-go/deepseek-v4-flash', 'opencode-go/glm-5.2', 'opencode-go/minimax-m3']
Provider backends: ['opencode-go']
API backends: ['hermes_cli', 'omp_cli', 'opencode_cli', 'opencode_go_chat']
Cohorts: ['agent_iterated / blank / opencode-go', 'agent_iterated / parsed / opencode-go', 'single_shot / exact / opencode-go']


## 1. Result integrity audit

This first checks the loaded fair table: raw merged rows, scored rows, excluded infrastructure rows, duplicate keys, missing artifacts, and judged failures. Do not interpret model rankings before this looks sane.


In [2]:
key_cols = ['harness', 'task', 'model', 'run']
keys = all_df[key_cols].astype(str)
duplicate_mask = keys.duplicated(keep=False)
expected = pd.MultiIndex.from_product(
    [sorted(all_df[c].dropna().astype(str).unique()) for c in key_cols],
    names=key_cols,
).to_frame(index=False)
actual_keys = keys.drop_duplicates()
missing = expected.merge(actual_keys, on=key_cols, how='left', indicator=True).query("_merge == 'left_only'").drop(columns='_merge')
path_missing = pd.DataFrame({
    'missing_workdir': [sum(bool(p) and not Path(str(p)).exists() for p in all_df['workdir'].fillna(''))],
    'missing_transcript': [sum(bool(p) and not Path(str(p)).exists() for p in all_df['transcript_path'].fillna(''))],
})
audit = pd.DataFrame({
    'raw_rows': [len(all_df)],
    'scored_rows': [len(df)],
    'excluded_rows': [len(excluded_df)],
    'expected_rectangular_rows': [len(expected)],
    'duplicate_keys': [int(duplicate_mask.sum())],
    'missing_keys': [len(missing)],
    'scored_infrastructure_errors': [int(df['errored'].sum())],
    'scored_task_failures': [int(df['failed'].sum())],
    'scored_successes': [int(df['passed'].sum())],
})
display(audit)
display(path_missing)
if len(excluded_df):
    display(excluded_df.groupby(['fair_status'])['task'].count().rename('excluded_rows').reset_index())
if len(missing):
    display(missing.head(50))


,raw_rows,scored_rows,excluded_rows,expected_rectangular_rows,duplicate_keys,missing_keys,scored_infrastructure_errors,scored_task_failures,scored_successes
0,396,275,121,396,0,0,0,25,250


,missing_workdir,missing_transcript
0,0,0


,fair_status,excluded_rows
0,excluded_infra,121


## 2. Coverage matrix

Coverage answers: what actually ran? This uses all merged rows, including rows excluded from fair model-quality scoring.


In [3]:
coverage = all_df.groupby(['harness', 'model_short']).size().unstack(fill_value=0)
display(coverage)

plt.figure(figsize=(max(7, coverage.shape[1] * 1.6), 3.5))
sns.heatmap(coverage, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Rows by harness × model — all merged rows')
plt.xlabel('Model')
plt.ylabel('Harness')
plt.tight_layout()
plt.show()


model_short,deepseek-v4-flash,glm-5.2,minimax-m3
harness,,,
hermes,33,33,33
omp,33,33,33
opencode,33,33,33
raw_api,33,33,33


/var/folders/03/pk9qb0rn42x1k34_3fs_rd4m0000gn/T/ipykernel_46856/2629029854.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Infrastructure exclusions

Rows excluded by `metrics_fair.csv` are not model quality failures. They indicate setup/baseline/tooling defects found by the audit/rescore pass.


In [4]:
if excluded_df.empty:
    print('No infrastructure exclusions in current result set.')
else:
    display(excluded_df.groupby(['harness', 'model_short', 'fair_status'])['task'].count().rename('excluded_rows').reset_index())
    display(excluded_df[['harness', 'task', 'model', 'adapter_model', 'fair_status', 'fair_notes', 'workdir', 'transcript_path']].head(100))


,harness,model_short,fair_status,excluded_rows
0,hermes,deepseek-v4-flash,excluded_infra,8
1,hermes,glm-5.2,excluded_infra,9
2,hermes,minimax-m3,excluded_infra,12
3,omp,deepseek-v4-flash,excluded_infra,9
4,omp,glm-5.2,excluded_infra,9
5,omp,minimax-m3,excluded_infra,10
6,opencode,deepseek-v4-flash,excluded_infra,8
7,opencode,glm-5.2,excluded_infra,9
8,opencode,minimax-m3,excluded_infra,11
9,raw_api,deepseek-v4-flash,excluded_infra,12


,harness,task,model,adapter_model,fair_status,fair_notes,workdir,transcript_path
36,hermes,ng-bug1-missing-input,deepseek/deepseek-v4-flash,opencode-go/deepseek-v4-flash,excluded_infra,Angular baseline lacks RealWorld theme asset,results/full_combined_v3/hermes__ng-bug1-missi...,results/full_combined_v3/hermes__ng-bug1-missi...
37,hermes,ng-bug1-missing-input,deepseek/deepseek-v4-flash,opencode-go/deepseek-v4-flash,excluded_infra,Angular baseline lacks RealWorld theme asset,results/full_combined_v3/hermes__ng-bug1-missi...,results/full_combined_v3/hermes__ng-bug1-missi...
39,hermes,ng-bug1-missing-input,minimax/minimax-m3,opencode-go/minimax-m3,excluded_infra,Angular baseline lacks ng executable,results/full_combined_v3/hermes__ng-bug1-missi...,results/full_combined_v3/hermes__ng-bug1-missi...
40,hermes,ng-bug1-missing-input,minimax/minimax-m3,opencode-go/minimax-m3,excluded_infra,Angular baseline lacks ng executable,results/full_combined_v3/hermes__ng-bug1-missi...,results/full_combined_v3/hermes__ng-bug1-missi...
41,hermes,ng-bug1-missing-input,minimax/minimax-m3,opencode-go/minimax-m3,excluded_infra,Angular baseline lacks ng executable,results/full_combined_v3/hermes__ng-bug1-missi...,results/full_combined_v3/hermes__ng-bug1-missi...
...,...,...,...,...,...,...,...,...
343,raw_api,ng-feat1-reading-time,deepseek/deepseek-v4-flash,opencode-go/deepseek-v4-flash,excluded_infra,Angular baseline lacks ng executable,results/full_combined_v3/raw_api__ng-feat1-rea...,results/full_combined_v3/raw_api__ng-feat1-rea...
344,raw_api,ng-feat1-reading-time,deepseek/deepseek-v4-flash,opencode-go/deepseek-v4-flash,excluded_infra,Angular baseline lacks ng executable,results/full_combined_v3/raw_api__ng-feat1-rea...,results/full_combined_v3/raw_api__ng-feat1-rea...
345,raw_api,ng-feat1-reading-time,minimax/minimax-m3,opencode-go/minimax-m3,excluded_infra,Angular baseline lacks ng executable,results/full_combined_v3/raw_api__ng-feat1-rea...,results/full_combined_v3/raw_api__ng-feat1-rea...
346,raw_api,ng-feat1-reading-time,minimax/minimax-m3,opencode-go/minimax-m3,excluded_infra,Angular baseline lacks ng executable,results/full_combined_v3/raw_api__ng-feat1-rea...,results/full_combined_v3/raw_api__ng-feat1-rea...


## 4. Comparison cohorts

Strict comparisons require the same capability mode, telemetry trust, and provider backend. Cross-cohort charts are descriptive only.


In [5]:
cohort_summary = df.groupby(['cohort', 'harness', 'model_short']).agg(
    rows=('passed', 'count'),
    pass_rate=('passed', 'mean'),
    error_rate=('errored', 'mean'),
    median_cost=('cost_usd', 'median'),
    median_latency=('latency_s', 'median'),
).reset_index().sort_values(['cohort', 'harness', 'model_short'])
display(cohort_summary.style.format({
    'pass_rate': '{:.0%}',
    'error_rate': '{:.0%}',
    'median_cost': '${:.4f}',
    'median_latency': '{:.1f}s',
}))


,cohort,harness,model_short,rows,pass_rate,error_rate,median_cost,median_latency
0,agent_iterated / blank / opencode-go,hermes,deepseek-v4-flash,25,100%,0%,$nan,31.5s
1,agent_iterated / blank / opencode-go,hermes,glm-5.2,24,92%,0%,$nan,22.6s
2,agent_iterated / blank / opencode-go,hermes,minimax-m3,21,100%,0%,$nan,26.9s
3,agent_iterated / parsed / opencode-go,omp,deepseek-v4-flash,24,100%,0%,$0.0022,26.5s
4,agent_iterated / parsed / opencode-go,omp,glm-5.2,24,83%,0%,$0.0688,22.9s
5,agent_iterated / parsed / opencode-go,omp,minimax-m3,23,91%,0%,$0.0039,29.7s
6,agent_iterated / parsed / opencode-go,opencode,deepseek-v4-flash,25,88%,0%,$0.0044,22.7s
7,agent_iterated / parsed / opencode-go,opencode,glm-5.2,24,75%,0%,$0.0822,23.1s
8,agent_iterated / parsed / opencode-go,opencode,minimax-m3,22,95%,0%,$0.0043,20.4s
9,single_shot / exact / opencode-go,raw_api,deepseek-v4-flash,21,90%,0%,$0.0002,7.6s


## 5. Within-cohort model performance

These charts are valid for comparing models inside each cohort. They do not compare `single_shot` raw API against agent harnesses.


In [6]:
for cohort_name, sub in df.groupby('cohort'):
    if sub.empty:
        continue
    pivot = sub.pivot_table(index='model_short', columns='task', values='passed', aggfunc='mean')
    plt.figure(figsize=(max(10, 0.65 * len(pivot.columns)), max(2.8, 0.45 * len(pivot.index))))
    sns.heatmap(pivot, vmin=0, vmax=1, cmap='RdYlGn', annot=True, fmt='.0%', cbar_kws={'label': 'pass rate'})
    plt.title(f'Pass rate by model × task — {cohort_name}')
    plt.xlabel('Task')
    plt.ylabel('Model')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


/var/folders/03/pk9qb0rn42x1k34_3fs_rd4m0000gn/T/ipykernel_46856/2000118737.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/03/pk9qb0rn42x1k34_3fs_rd4m0000gn/T/ipykernel_46856/2000118737.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/03/pk9qb0rn42x1k34_3fs_rd4m0000gn/T/ipykernel_46856/2000118737.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Harness overview

Descriptive only. This chart intentionally includes `raw_api`, `omp`, `opencode`, and `hermes` together, but their capability modes and telemetry trust differ.


In [7]:
harness_order = [h for h in ['raw_api', 'omp', 'opencode', 'hermes'] if h in set(df['harness'])]
overview = df.groupby(['harness', 'model_short']).agg(
    pass_rate=('passed', 'mean'),
    rows=('passed', 'count'),
    error_rate=('errored', 'mean'),
).reset_index()
plt.figure(figsize=(max(8, 1.7 * overview['model_short'].nunique()), 5))
sns.barplot(data=overview, x='model_short', y='pass_rate', hue='harness', hue_order=harness_order)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.title('All-harness pass-rate overview — descriptive, not strict ranking')
plt.xlabel('Canonical model')
plt.ylabel('Pass rate')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Harness')
plt.tight_layout()
plt.show()

display(overview.sort_values(['harness', 'model_short']).style.format({'pass_rate': '{:.0%}', 'error_rate': '{:.0%}'}))


/var/folders/03/pk9qb0rn42x1k34_3fs_rd4m0000gn/T/ipykernel_46856/324914959.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,harness,model_short,pass_rate,rows,error_rate
0,hermes,deepseek-v4-flash,100%,25,0%
1,hermes,glm-5.2,92%,24,0%
2,hermes,minimax-m3,100%,21,0%
3,omp,deepseek-v4-flash,100%,24,0%
4,omp,glm-5.2,83%,24,0%
5,omp,minimax-m3,91%,23,0%
6,opencode,deepseek-v4-flash,88%,25,0%
7,opencode,glm-5.2,75%,24,0%
8,opencode,minimax-m3,95%,22,0%
9,raw_api,deepseek-v4-flash,90%,21,0%


## 7. Cost, latency, and model calls

Only compare cost/token numbers inside cohorts with the same telemetry trust and provider backend. `pricing_model` records the price table key used for `cost_usd`.


In [8]:
cost_rows = df[df['cost_usd'].notna() & (df['cost_usd'] > 0)].copy()
if cost_rows.empty:
    print('No positive cost rows.')
else:
    display(cost_rows.groupby(['cohort', 'harness', 'model_short', 'pricing_model']).agg(
        rows=('passed', 'count'),
        median_cost=('cost_usd', 'median'),
        median_latency=('latency_s', 'median'),
        median_calls=('model_calls', 'median'),
    ).reset_index().style.format({
        'median_cost': '${:.4f}',
        'median_latency': '{:.1f}s',
        'median_calls': '{:.1f}',
    }))

    g = sns.relplot(
        data=cost_rows,
        x='latency_s', y='cost_usd', hue='harness', style='provider_backend',
        col='model_short', col_wrap=3, height=3.2, facet_kws={'sharex': False, 'sharey': False},
    )
    g.fig.suptitle('Cost vs latency by model — inspect within cohort only', y=1.03)
    for ax in g.axes.flat:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y:.3f}'))
    plt.show()


,cohort,harness,model_short,pricing_model,rows,median_cost,median_latency,median_calls
0,agent_iterated / parsed / opencode-go,omp,deepseek-v4-flash,opencode-go/deepseek-v4-flash,24,$0.0022,26.5s,6.0
1,agent_iterated / parsed / opencode-go,omp,glm-5.2,opencode-go/glm-5.2,24,$0.0688,22.9s,5.0
2,agent_iterated / parsed / opencode-go,omp,minimax-m3,opencode-go/minimax-m3,23,$0.0039,29.7s,8.0
3,agent_iterated / parsed / opencode-go,opencode,deepseek-v4-flash,opencode-go/deepseek-v4-flash,25,$0.0044,22.7s,5.0
4,agent_iterated / parsed / opencode-go,opencode,glm-5.2,opencode-go/glm-5.2,24,$0.0822,23.1s,4.0
5,agent_iterated / parsed / opencode-go,opencode,minimax-m3,opencode-go/minimax-m3,22,$0.0043,20.4s,4.0
6,single_shot / exact / opencode-go,raw_api,deepseek-v4-flash,opencode-go/deepseek-v4-flash,21,$0.0002,7.6s,1.0
7,single_shot / exact / opencode-go,raw_api,glm-5.2,opencode-go/glm-5.2,21,$0.0041,7.0s,1.0
8,single_shot / exact / opencode-go,raw_api,minimax-m3,opencode-go/minimax-m3,21,$0.0011,8.1s,1.0


/var/folders/03/pk9qb0rn42x1k34_3fs_rd4m0000gn/T/ipykernel_46856/265831649.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Baseline overlap

Set `BASELINE_RESULTS` in the first cell to compare another result directory. The comparison only uses overlapping canonical models and tasks, then still requires provider/cohort review before drawing conclusions.


In [9]:
if baseline_df.empty:
    print('No baseline configured. Set BASELINE_RESULTS = Path(...) in the first cell to enable this section.')
else:
    overlap_models = sorted(set(df['model']) & set(baseline_df['model']))
    overlap_tasks = sorted(set(df['task']) & set(baseline_df['task']))
    joined = pd.concat([
        df[df['model'].isin(overlap_models) & df['task'].isin(overlap_tasks)],
        baseline_df[baseline_df['model'].isin(overlap_models) & baseline_df['task'].isin(overlap_tasks)],
    ])
    summary = joined.groupby(['result_set', 'harness', 'model_short', 'provider_backend']).agg(
        rows=('passed', 'count'),
        pass_rate=('passed', 'mean'),
        error_rate=('errored', 'mean'),
    ).reset_index()
    display(summary.style.format({'pass_rate': '{:.0%}', 'error_rate': '{:.0%}'}))


No baseline configured. Set BASELINE_RESULTS = Path(...) in the first cell to enable this section.


## 9. Failure drilldowns

Task failures are the useful debugging surface after infrastructure exclusions and post-hoc rescoring have been applied.


In [10]:
failures = df[df['failed'] & ~df['errored']].copy()
if failures.empty:
    print('No judged task failures.')
else:
    task_rank = df.groupby('task').agg(
        rows=('passed', 'count'),
        pass_rate=('passed', 'mean'),
        failure_rate=('failed', 'mean'),
    ).sort_values(['failure_rate', 'task'], ascending=[False, True])
    display(task_rank.style.format({'pass_rate': '{:.0%}', 'failure_rate': '{:.0%}'}))
    display(failures[['harness', 'task', 'model', 'adapter_model', 'fair_build_ok', 'fair_test_ok', 'fair_status', 'fair_notes', 'workdir', 'transcript_path']].head(100))


,rows,pass_rate,failure_rate
task,,,
sb-feat1-name-length,21,52%,48%
data-feat1-customer-ranking,36,78%,22%
re-feat2-author-filter,36,81%,19%
bug1-petvalidator,36,100%,0%
bug2-ownercontroller,36,100%,0%
data-bug1-sales-genre,36,100%,0%
ng-bug1-missing-input,1,100%,0%
ng-feat2-service-search,1,100%,0%
re-bug1-favorite-count,36,100%,0%


,harness,task,model,adapter_model,fair_build_ok,fair_test_ok,fair_status,fair_notes,workdir,transcript_path
34,hermes,data-feat1-customer-ranking,z-ai/glm-5.2,opencode-go/glm-5.2,True,False,task.semantic_failure,verifier assertion failed,results/full_combined_v3/hermes__data-feat1-cu...,results/full_combined_v3/hermes__data-feat1-cu...
35,hermes,data-feat1-customer-ranking,z-ai/glm-5.2,opencode-go/glm-5.2,True,False,task.semantic_failure,verifier assertion failed,results/full_combined_v3/hermes__data-feat1-cu...,results/full_combined_v3/hermes__data-feat1-cu...
132,omp,data-feat1-customer-ranking,z-ai/glm-5.2,opencode-go/glm-5.2,True,False,task.semantic_failure,verifier assertion failed,results/full_combined_v3/omp__data-feat1-custo...,results/full_combined_v3/omp__data-feat1-custo...
133,omp,data-feat1-customer-ranking,z-ai/glm-5.2,opencode-go/glm-5.2,True,False,task.semantic_failure,verifier assertion failed,results/full_combined_v3/omp__data-feat1-custo...,results/full_combined_v3/omp__data-feat1-custo...
134,omp,data-feat1-customer-ranking,z-ai/glm-5.2,opencode-go/glm-5.2,True,False,task.semantic_failure,verifier assertion failed,results/full_combined_v3/omp__data-feat1-custo...,results/full_combined_v3/omp__data-feat1-custo...
192,omp,sb-feat1-name-length,minimax/minimax-m3,opencode-go/minimax-m3,False,False,task_or_harness_failure,unclassified failure; inspect artifacts,results/full_combined_v3/omp__sb-feat1-name-le...,results/full_combined_v3/omp__sb-feat1-name-le...
194,omp,sb-feat1-name-length,minimax/minimax-m3,opencode-go/minimax-m3,False,False,task_or_harness_failure,unclassified failure; inspect artifacts,results/full_combined_v3/omp__sb-feat1-name-le...,results/full_combined_v3/omp__sb-feat1-name-le...
195,omp,sb-feat1-name-length,z-ai/glm-5.2,opencode-go/glm-5.2,False,False,task_or_harness_failure,unclassified failure; inspect artifacts,results/full_combined_v3/omp__sb-feat1-name-le...,results/full_combined_v3/omp__sb-feat1-name-le...
231,opencode,data-feat1-customer-ranking,z-ai/glm-5.2,opencode-go/glm-5.2,True,False,task.semantic_failure,verifier assertion failed,results/full_combined_v3/opencode__data-feat1-...,results/full_combined_v3/opencode__data-feat1-...
232,opencode,data-feat1-customer-ranking,z-ai/glm-5.2,opencode-go/glm-5.2,True,False,task.semantic_failure,verifier assertion failed,results/full_combined_v3/opencode__data-feat1-...,results/full_combined_v3/opencode__data-feat1-...


## 10. Summary table

Compact per-cohort summary. Use this table for decisions only after Sections 1–4 look clean.


In [11]:
final = df.groupby(['cohort', 'harness', 'model_short', 'provider_backend', 'api_backend']).agg(
    runs=('passed', 'count'),
    pass_rate=('passed', 'mean'),
    error_rate=('errored', 'mean'),
    median_cost=('cost_usd', 'median'),
    median_latency=('latency_s', 'median'),
    median_in_tok=('in_tok', 'median'),
    median_out_tok=('out_tok', 'median'),
).sort_values(['cohort', 'pass_rate'], ascending=[True, False])
final.style.format({
    'pass_rate': '{:.0%}',
    'error_rate': '{:.0%}',
    'median_cost': '${:.4f}',
    'median_latency': '{:.1f}s',
    'median_in_tok': '{:.0f}',
    'median_out_tok': '{:.0f}',
})
